# 02 · A/B Experiment Analysis

We analyse the primary comparison — **Mens E-Mail vs No E-Mail** — the way it
should be done in production:

1. compute the sample size the experiment *would* have needed, up front;
2. check for **sample-ratio mismatch (SRM)** before trusting anything;
3. report **lift with a 95% confidence interval**, not just a p-value;
4. correct for testing **multiple metrics** (Bonferroni);
5. show how **CUPED** cuts variance using a pre-experiment covariate.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
FIG = Path.cwd().parent / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

In [2]:
from src.data.load import load_raw, add_treatment_flag
from src.experiment.ab import (
    calculate_sample_size, two_proportion_ztest, sample_ratio_mismatch,
    cuped_adjust, bonferroni,
)
two = add_treatment_flag(load_raw())
control = two[two.treatment == 0]; treat = two[two.treatment == 1]
len(control), len(treat)

(21306, 21307)

## 1. Sample size (computed *before* looking at results)

Baseline visit rate in control is ~10.6%. How many customers per arm would we
need to detect a **20% relative lift** at 80% power, 5% significance?

In [3]:
baseline = control.visit.mean()
n_needed = calculate_sample_size(baseline_rate=baseline, mde=0.20, alpha=0.05, power=0.8)
print(f"baseline visit rate      : {baseline:.4f}")
print(f"required n per arm (MDE 20%): {n_needed:,}")
print(f"actual n per arm          : {len(treat):,}")
print("Powered:", len(treat) >= n_needed)

baseline visit rate      : 0.1062
required n per arm (MDE 20%): 1,794
actual n per arm          : 21,307
Powered: True


## 2. Sample-ratio mismatch
A broken split invalidates everything downstream.

In [4]:
srm = sample_ratio_mismatch(len(control), len(treat))
srm

{'observed_ratio': 0.4999882664914463,
 'chi2': 2.346701710745547e-05,
 'p_value': np.float64(0.9961348415003554),
 'srm_detected': np.False_}

## 3. Effect on each metric — lift + 95% CI

In [5]:
def rate_test(col):
    return two_proportion_ztest(
        {"conversions": int(control[col].sum()), "visitors": len(control)},
        {"conversions": int(treat[col].sum()),   "visitors": len(treat)},
    )
res_visit = rate_test("visit")
res_conv  = rate_test("conversion")
print("VISIT      ", res_visit)
print("CONVERSION ", res_conv)

VISIT       control=0.1062 treatment=0.1828 lift=+72.1% p=0.0000 CI95=[+0.0700, +0.0832] (significant)
CONVERSION  control=0.0057 treatment=0.0125 lift=+118.8% p=0.0000 CI95=[+0.0050, +0.0086] (significant)


### Spend is continuous — use a Welch t-test

Spend is mostly zero with a long right tail, so a proportion test does not
apply. We report the difference in mean spend with a t-test.

In [6]:
from scipy import stats
t_stat, p_spend = stats.ttest_ind(treat.spend, control.spend, equal_var=False)
diff = treat.spend.mean() - control.spend.mean()
se = np.sqrt(treat.spend.var()/len(treat) + control.spend.var()/len(control))
ci = (diff - 1.96*se, diff + 1.96*se)
print(f"control mean spend  : {control.spend.mean():.4f}")
print(f"treat mean spend    : {treat.spend.mean():.4f}")
print(f"diff                : {diff:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]  p={p_spend:.4f}")

control mean spend  : 0.6528
treat mean spend    : 1.4226
diff                : +0.7698  95% CI [+0.4851, +1.0545]  p=0.0000


## 4. Multiple-comparison correction

We tested three metrics. Under a family-wise error rate of 5%, each test must
clear a stricter Bonferroni threshold.

In [7]:
alpha_corr = bonferroni(0.05, 3)
print(f"Bonferroni alpha (3 tests): {alpha_corr:.4f}")
for name, r in [("visit", res_visit), ("conversion", res_conv), ("spend", None)]:
    p = p_spend if r is None else r.p_value
    print(f"  {name:11s} p={p:.4g}  significant @ corrected alpha: {p < alpha_corr}")

Bonferroni alpha (3 tests): 0.0167
  visit       p=0  significant @ corrected alpha: True
  conversion  p=1.523e-13  significant @ corrected alpha: True
  spend       p=1.164e-07  significant @ corrected alpha: True


## 5. CUPED — variance reduction with a pre-experiment covariate

CUPED removes variance explained by a pre-treatment covariate (here `history`,
the customer's prior 12-month spend) *without bias*, which in a real experiment
means reaching significance with fewer samples or less time.

$$y_{\text{cuped}} = y - \theta\,(x - \bar{x}), \qquad \theta = \frac{\text{cov}(y, x)}{\text{var}(x)}$$

In [8]:
y_adj, vr = cuped_adjust(two.spend.values, two.history.values)
print(f"variance reduction on spend via CUPED: {vr:.2%}")
print(f"raw spend variance     : {np.var(two.spend.values, ddof=1):,.2f}")
print(f"CUPED spend variance   : {np.var(y_adj, ddof=1):,.2f}")

variance reduction on spend via CUPED: 0.04%
raw spend variance     : 224.89
CUPED spend variance   : 224.80


## Takeaways

- The experiment is **well powered** and shows **no SRM** — safe to trust.
- Mens E-Mail drives a large, highly significant lift in **visits** and a
  smaller but significant lift in **conversion** and **spend**, all surviving
  Bonferroni correction.
- CUPED with prior-spend as covariate removes only a little variance here — the
  two-week spend outcome is weakly related to the pre-period covariate. This is
  itself a useful finding: CUPED pays off only when the covariate genuinely
  predicts the metric.
- **But** an average effect does not tell us *who* to e-mail. A positive
  average can hide customers the e-mail annoys into not buying. That is the
  causal question notebook 03 answers with uplift modeling.